# v2.3.6 대분류별 자료 선택 검증

실제 국회도서관 API·OpenAI 호출 없이 **합성 자료**로 선택 과정을 확인합니다.
노트북을 `app.py`가 있는 프로젝트 폴더에서 순서대로 실행하세요.
각 실습 뒤의 별도 `assert` 셀이 요구사항 충족 여부를 확인합니다.
화면 디자인·실제 API 응답 검증과는 구분합니다.

## 실습 1. 테스트용 자료 준비

In [1]:
from pathlib import Path
import pandas as pd
from catalog_identity import identity_rows, ROWKEY, KIND, TITLE, CONTROL
from nanet_metadata import THESIS, ARTICLE, BOOK, UNKNOWN
from catalog_picker import (
    category_counts, category_items, category_label, record_label,
    prepare_category, prepare_record, remember_record, clear_picker,
    reading_target, CATEGORY_WIDGET, RECORD_WIDGET, RECORD_MEMORY,
)

# 실제 논문이 아닌 합성 서지정보입니다.
df = pd.DataFrame([
    {'controlno': 'KDMT2026000001', 'title': '테스트 학위논문 A', 'author': '예시저자 A', 'year': 2026},
    {'controlno': 'KINX2026000002', 'title': '테스트 학술기사', 'author': '예시저자 B', 'year': 2025},
    {'controlno': 'MONO2026000003', 'title': '테스트 참고도서', 'author': '예시저자 C', 'year': 2024},
    {'controlno': 'KDMT2026000004', 'title': '테스트 학위논문 B', 'author': '예시저자 D', 'year': 2023},
    {'controlno': 'UNKN2026000005', 'title': '테스트 미확인 자료'},
])
items = identity_rows(df)
counts = category_counts(items)
print('현재 수집 자료:', len(items), '건')
print('유형별 건수:', counts)

현재 수집 자료: 5 건
유형별 건수: {'학위논문': 2, '학술기사': 1, '도서자료': 1, '자료유형 미확인': 1}


In [2]:
# 검증 1: 실제 수집 자료와 분류별 건수 일치
assert len(items) == 5
assert counts == {THESIS: 2, ARTICLE: 1, BOOK: 1, UNKNOWN: 1}
assert sum(counts.values()) == len(items)
print('검증 1 통과')

검증 1 통과


## 실습 2. 대분류 선택 후 세부 목록 만들기

In [3]:
# 학위논문을 선택했을 때 학위논문만 표시합니다.
state = {}
category, counts = prepare_category(state, items)
options = prepare_record(state, items, category)
print('기본 대분류:', category_label(category, counts))
for item in options:
    print(record_label(item))

기본 대분류: 🎓 학위논문 · 2건
테스트 학위논문 A | 예시저자 A | 2026 | KDMT2026000001
테스트 학위논문 B | 예시저자 D | 2023 | KDMT2026000004


In [4]:
# 검증 2: 자동 분류를 수정하지 않고 표시 목록만 필터링
assert category == THESIS
assert len(options) == 2
assert all(item[KIND] == THESIS for item in options)
assert state[RECORD_WIDGET] is None  # 사용자가 자료를 고르기 전
assert len(items) == 5
print('검증 2 통과')

검증 2 통과


## 실습 3. 자료 선택과 분류 전환

In [5]:
# 두 번째 학위논문 선택
thesis_key = options[1][ROWKEY]
state[RECORD_WIDGET] = thesis_key
remember_record(state, THESIS, thesis_key)

# 학술기사로 바꾸면 이전 학위논문을 자동 선택하지 않습니다.
state[CATEGORY_WIDGET] = ARTICLE
category, _ = prepare_category(state, items)
article_options = prepare_record(state, items, category)
selection_after_change = state[RECORD_WIDGET]
print('변경된 대분류:', category)
print('해당 자료:', [record_label(item) for item in article_options])

# 다시 학위논문으로 돌아오면 해당 분류에서 고른 자료를 복원합니다.
state[CATEGORY_WIDGET] = THESIS
category, _ = prepare_category(state, items)
prepare_record(state, items, category)
print('복원된 내부 식별자:', state[RECORD_WIDGET])

변경된 대분류: 학술기사
해당 자료: ['테스트 학술기사 | 예시저자 B | 2025 | KINX2026000002']
복원된 내부 식별자: control:KDMT2026000004


In [6]:
# 검증 3: 다른 유형 선택이 잘못 남지 않고 유형별 기억을 유지
assert selection_after_change is None
assert len(article_options) == 1 and article_options[0][KIND] == ARTICLE
assert state[RECORD_WIDGET] == thesis_key
print('검증 3 통과')

검증 3 통과


## 실습 4. 다음 페이지를 추가하거나 제목을 보완한 경우

In [7]:
# 행 순서 변경·제목 보완·자료 추가를 가정합니다.
updated = df.iloc[::-1].copy()
updated.loc[updated['controlno'] == 'KDMT2026000004', 'title'] = '보완된 테스트 학위논문 B'
updated = pd.concat([updated, pd.DataFrame([
    {'controlno':'KDMT2026000006', 'title':'추가된 테스트 학위논문'}
])], ignore_index=True)
new_items = identity_rows(updated)
category, _ = prepare_category(state, new_items)
new_options = prepare_record(state, new_items, category)
selected = next(item for item in new_options if item[ROWKEY] == state[RECORD_WIDGET])
print('현재 선택:', selected[CONTROL], selected[TITLE])

현재 선택: KDMT2026000004 보완된 테스트 학위논문 B


In [8]:
# 검증 4: 행번호가 아닌 자료 식별자로 선택 유지
assert state[RECORD_WIDGET] == thesis_key
assert selected[CONTROL] == 'KDMT2026000004'
assert selected[TITLE] == '보완된 테스트 학위논문 B'
assert len(new_options) == 3
print('검증 4 통과')

검증 4 통과


## 실습 5. 도서·미확인 자료 전달 제한 및 새 검색

In [9]:
thesis = category_items(items, THESIS)[0]
article = category_items(items, ARTICLE)[0]
book = category_items(items, BOOK)[0]
unknown = category_items(items, UNKNOWN)[0]
print('학위논문 이동 대상:', reading_target(thesis))
print('학술기사 이동 대상:', reading_target(article))
print('도서 이동 대상:', reading_target(book) or '이동 제한: 참고자료')
print('미확인 자료 이동 대상:', reading_target(unknown) or '이동 제한: 분류 미확인')

# 새 검색의 선택값만 지우고 기존 연구노트는 남깁니다.
state['research_library'] = [{'note':'사용자가 작성한 연구 질문'}]
clear_picker(state)

학위논문 이동 대상: KDMT2026000001
학술기사 이동 대상: KINX2026000002
도서 이동 대상: 이동 제한: 참고자료
미확인 자료 이동 대상: 이동 제한: 분류 미확인


In [10]:
# 검증 5: 잘못된 논문 전달 방지·사용자 연구노트 보존
assert reading_target(thesis) == 'KDMT2026000001'
assert reading_target(article) == 'KINX2026000002'
assert reading_target(book) == ''
assert reading_target(unknown) == ''
assert CATEGORY_WIDGET not in state and RECORD_MEMORY not in state
assert state['research_library'][0]['note'] == '사용자가 작성한 연구 질문'
print('검증 5 통과 — 모든 별도 검증 셀을 통과했습니다.')

검증 5 통과 — 모든 별도 검증 셀을 통과했습니다.


## 실제 앱에서 추가 확인
`python -m streamlit run app.py`로 실행한 뒤 대분류와 자료를 직접 선택하세요.
콤보박스 클릭·강조 색상·좁은 화면 줄바꿈·02 이동은 브라우저에서 별도로 확인해야 합니다.
이 노트북은 실시간 검색 성공이나 UI 렌더링을 보증하지 않습니다.